# Stereo-aware InChIKey catalogue planning: Boceprevir

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laboratoire-de-Chemoinformatique/SynPlanner/blob/main/tutorials/20_InChIKey_Building_Block_Catalogue.ipynb)

This tutorial uses an **already prepared**, vendor-aware JSON building-block catalogue. It shows the two immutable indexes used by MCTS, a stereo-aware search for Boceprevir, and vendor cost calculation on a detached Route. It does not prepare or modify the catalogue.

In [ ]:
# Colab setup — this cell does nothing when you run the notebook locally.
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [
            "pip",
            "install",
            "-q",
            "git+https://github.com/Laboratoire-de-Chemoinformatique/SynPlanner.git@main",
        ],
        check=True,
    )
    print("SynPlanner installed. If an import below fails, restart the runtime.")


## 1. Load the processed catalogue

Set SYNPLAN_BUILDING_BLOCKS_JSON to the prepared JSON file, or place it at the default path below. In Colab, upload or mount the file first. The GPS preset supplies the trained ranking policy and reaction rules, but its legacy SMILES stock is deliberately not used here.

In [ ]:
import os
from pathlib import Path

from synplan.utils.loading import download_selected_files

catalogue_path = Path(
    os.environ.get(
        "SYNPLAN_BUILDING_BLOCKS_JSON",
        "synplan_data/building_blocks/all-bb-2026-06/building_blocks.json",
    )
).expanduser()
if not catalogue_path.is_file():
    raise FileNotFoundError(
        f"Prepared InChIKey catalogue not found at {catalogue_path}. "
        "Set SYNPLAN_BUILDING_BLOCKS_JSON or create the JSON first with "
        "'synplan building_blocks_standardizing --input <catalogue.tsv> "
        "--output <catalogue.json>'."
    )

# These are the reaction_rules and ranking_policy entries declared by the
# current synplanner-gps preset. Its other assets, especially its legacy
# building-block stock, are intentionally not downloaded.
data_root = download_selected_files(
    [
        ("policy/supervised_gps/v1", "reaction_rules.tsv"),
        ("policy/supervised_gps/v1/v1", "ranking_policy.ckpt"),
    ],
    save_to="synplan_data",
    extract_zips=False,
)
reaction_rules_path = data_root / "policy/supervised_gps/v1/reaction_rules.tsv"
ranking_policy_path = (
    data_root / "policy/supervised_gps/v1/v1/ranking_policy.ckpt"
)

In [ ]:
from synplan.chem.building_blocks import load_building_block_indexes

building_blocks_by_inchikey, building_block_candidates = (
    load_building_block_indexes(catalogue_path)
)

{
    "full_inchikey_records": len(building_blocks_by_inchikey),
    "connectivity_buckets": len(building_block_candidates),
}

In [ ]:
example_key, example_block = next(iter(building_blocks_by_inchikey.items()))
example_bucket = building_block_candidates[example_key[:14]]

{
    "inchikey": example_key,
    "smiles": example_block.smiles,
    "vendors": dict(example_block.vendors),
    "has_stereo": example_block.has_stereo,
    "candidate_count_for_first_14_characters": len(example_bucket),
}

## 2. Preserve Boceprevir stereochemistry

Boceprevir contains explicit tetrahedral stereochemistry. clean_stereo=False is therefore essential: it makes the search select full InChIKey matching once, at tree construction, and retain that policy for all MCTS stock checks.

In [ ]:
from IPython.display import display

from synplan.chem.building_blocks import molecule_has_stereo, molecule_to_inchikey
from synplan.chem.utils import mol_from_smiles

BOCEPREVIR_SMILES = (
    "CC1([C@@H]2[C@H]1[C@H](N(C2)C(=O)[C@H](C(C)(C)C)"
    "NC(=O)NC(C)(C)C)C(=O)NC(CC3CCC3)C(=O)C(=O)N)C"
)
BOCEPREVIR_INCHIKEY = "LHHCSNFAOIFYRV-DOVBMPENSA-N"

target_molecule = mol_from_smiles(BOCEPREVIR_SMILES, clean_stereo=False)
target_inchikey = molecule_to_inchikey(target_molecule)

assert molecule_has_stereo(target_molecule)
assert target_inchikey == BOCEPREVIR_INCHIKEY
display(target_molecule)
{
    "name": "Boceprevir",
    "canonical_smiles": str(target_molecule),
    "full_inchikey": target_inchikey,
    "has_stereo": molecule_has_stereo(target_molecule),
}

## 3. Run stereo-aware MCTS

Both the tree and rollout evaluator receive the complete-key map and the prefix candidate index. The ranking model and rules are the normal GPS planning resources; only stock identity changes.

In [ ]:
from synplan.utils.loading import load_policy_function, load_reaction_rules

reaction_rules = load_reaction_rules(reaction_rules_path)
policy_function = load_policy_function(weights_path=ranking_policy_path)

In [ ]:
from synplan.mcts.config import RolloutEvaluationConfig, TreeConfig
from synplan.utils.loading import load_evaluation_function

tree_config = TreeConfig(
    search_strategy="expansion_first",
    max_iterations=300,
    max_time=120,
    max_depth=9,
    min_mol_size=1,
    init_node_value=0.5,
    ucb_type="uct",
    c_ucb=0.1,
)

evaluation_config = RolloutEvaluationConfig(
    policy_network=policy_function,
    reaction_rules=reaction_rules,
    building_blocks=building_blocks_by_inchikey,
    building_block_candidates=building_block_candidates,
    min_mol_size=tree_config.min_mol_size,
    max_depth=tree_config.max_depth,
    normalize=True,
)
evaluation_function = load_evaluation_function(evaluation_config)

In [ ]:
from synplan.mcts.tree import Tree

tree = Tree(
    target=target_molecule,
    config=tree_config,
    reaction_rules=reaction_rules,
    building_blocks=building_blocks_by_inchikey,
    building_block_candidates=building_block_candidates,
    expansion_function=policy_function,
    evaluation_function=evaluation_function,
)

assert tree.use_full_inchikey
tree.run()

## 4. Select and cost a Route

Solved routes are preferred. A bounded search can finish before solving Boceprevir; in that case the best unfinished route is still a valid detached Route, and costing it explicitly reports missing or unpriced leaves instead of inventing a complete total.

In [ ]:
solved_routes = tree.routes()
if solved_routes:
    route_status = "solved"
    selected_route = solved_routes[0]
else:
    unfinished_routes = tree.routes(solved_only=False)
    if not unfinished_routes:
        raise RuntimeError("The bounded search did not produce an expandable route.")
    route_status = "unfinished"
    selected_route = unfinished_routes[0]

{
    "route_status": route_status,
    "tree_node_id": selected_route.provenance.tree_node_id,
    "steps": len(selected_route),
    "leaves": len(selected_route.leaves()),
}

In [ ]:
selected_route

In [ ]:
leaf_matches = []
for leaf in selected_route.leaves():
    leaf_key = molecule_to_inchikey(leaf)
    leaf_matches.append(
        {
            "smiles": str(leaf),
            "full_inchikey": leaf_key,
            "exact_catalogue_match": leaf_key in building_blocks_by_inchikey,
            "prefix_candidate_count": len(
                building_block_candidates.get(leaf_key[:14], ())
            ),
        }
    )

leaf_matches

In [ ]:
import json

cost = selected_route.calculate_cost(building_block_candidates)
json.dumps(cost)  # Route costs are ready for a JSON sidecar.
cost_label = (
    "complete solved-route cost"
    if route_status == "solved"
    else "incomplete cost for the best unfinished route"
)

{
    "route_status": route_status,
    "cost_label": cost_label,
    "complete_cost": cost["complete"],
    "cost_per_mol": cost["cost_per_mol"],
    "cost_per_gram": cost["cost_per_gram"],
    "priced_cost_per_mol": cost["priced_cost_per_mol"],
    "priced_cost_per_gram": cost["priced_cost_per_gram"],
    "cost_units": cost["cost_units"],
    "missing_leaves": cost["missing_leaves"],
    "unpriced_leaves": cost["unpriced_leaves"],
}

In [ ]:
import pandas as pd

pd.DataFrame(cost["leaves"])

## Takeaways

- The JSON is keyed by full Chython Standard InChIKeys; it stores canonical SMILES, stereo presence, and positive vendor prices.
- Boceprevir has explicit stereo, so Tree.use_full_inchikey is True and every stock check requires a complete-key match.
- A target without explicit atom or bond stereo would use the first 14 characters and consider every record in that candidate bucket.
- Route.calculate_cost(building_block_candidates) does not mutate the route or retain the catalogue. Missing and unpriced leaves keep the result honest when a route is unfinished or the catalogue is incomplete.
- Costing assumes one molar equivalent per leaf occurrence and 100% reaction yield. Catalogue prices remain unnormalized raw price-per-gram values, so this is a comparable material estimate rather than process economics.